<a href="https://colab.research.google.com/github/yajima-yasutoshi/Model2026/blob/main/20260617/%E3%83%8A%E3%83%83%E3%83%97%E3%82%B6%E3%83%83%E3%82%AF%E5%95%8F%E9%A1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#準備

In [ ]:
%%capture
# python-mip パッケージのインストール
!pip install mip

# ナップザック問題

本講義では、数理計画法の代表的な問題の一つである「ナップザック問題」を取り上げます。ナップザック問題は、そのシンプルさにもかかわらず、資源配分、プロジェクト選択、投資ポートフォリオ作成など、実社会における多様な意思決定問題の基礎となる重要な問題です。

この講義を通じて、以下の内容を学ぶことを目標とします。

* ナップザック問題の構造と特徴を理解する。
* ナップザック問題を数理モデルとして定式化する方法を習得する。
* Pythonの数理最適化ライブラリ `python-mip` を用いて、ナップザック問題を実際に解くプログラミングスキルを身につける。

## ナップザック問題の定義

ナップザック問題は、以下のように定義される組み合わせ最適化問題である。

**状況:**
* あなたは限られた容量を持つナップザック（リュックサック）を持っている。
* それぞれ価値と重さが異なる複数のアイテムがある。

**目的:**
* ナップザックの容量を超えないようにアイテムを選び、ナップザックに入れる。
* 選んだアイテムの総価値が最大になるようにする。

最も基本的な形式は **0-1 ナップザック問題** と呼ばれ、各アイテムは「ナップザックに入れる（1）」か「入れない（0）」かのどちらか一方を選択する。各アイテムは1つしか存在しないと仮定する。

**実社会での応用例:**
* **資源配分:** 限られた予算や人員、設備といった資源を、最大の効果が得られるように様々なプロジェクトやタスクに割り当てる問題。各プロジェクトがアイテムに、予算や人員が重さに、期待される効果が価値に対応する。
* **投資選択:** 限られた資金でどの株式や債券に投資するかを決定し、期待収益を最大化する問題。各金融商品がアイテムに、投資額が重さに、期待収益が価値に対応する。
* **貨物積載:** トラックや船舶などの輸送手段に、積載容量の制限内で最も価値の高い貨物を積む組み合わせを選ぶ問題。
* **プロジェクト選択:** 企業が限られたリソース（予算、時間、人員）の中で、最も利益貢献度の高いプロジェクト群を選択する問題。

## 数理モデル

ナップザック問題を数学的に記述し、最適化問題を解くための準備を行う。

**パラメータ:**
* $N$: アイテムの総数
* $v_i$: アイテム $i$ の価値 ($i = 1, \dots, N$)
* $w_i$: アイテム $i$ の重さ ($i = 1, \dots, N$)
* $C$: ナップザックの最大容量

**決定変数:**
各アイテム $i$ をナップザックに入れるかどうかを決定する
変数 $x_i$ を定義する。
ただし、この変数は0か1のどちらかの値を取るものとする。
バイナリ（binary）変数と呼ばれる。

$$x_i = \begin{cases} 1 & \text{アイテム } i \text{ をナップザックに入れる場合} \\ 0 & \text{アイテム } i \text{ をナップザックに入れない場合} \end{cases} \quad (i = 1, \dots, N)$$

**目的関数:**
ナップザックに入れられたアイテムの総価値を最大化する。
$$
\text{Maximize} \quad Z = v_1 x_1 + v_2 x_2 + \cdots + v_N x_N = \sum_{i=1}^{N} v_i x_i
$$

**制約条件:**
ナップザックに入れられたアイテムの総重量は、ナップザックの最大容量 $C$ を超えてはならない。
$$
w_1 x_1 + w_2 x_2 + \cdots + w_N x_N \le C
$$
この数式は和の記号を使い以下のように記述できる。
$$
\sum_{i=1}^{N} w_i x_i \le C
$$

**変数の定義域:**
決定変数はバイナリ変数である。
$$
x_i \in \{0, 1\} \quad (i = 1, \dots, N)
$$

このような定式化は、
変数に整数性（この問題の場合は0または1）の条件があることから
**整数線形計画問題**と呼ばれる


## Python MIP を用いた実装

次に、`python-mip` パッケージを用いてナップザック問題を解く。

例題として、以下のアイテムデータとナップザック容量で、
総価値の最大化を考える。

* **アイテムの価値と重さ:**
    * 価値: ` [60, 100, 120, 80, 90]`
    * 重さ: `[10, 20, 30, 15, 25]`
* **ナップザック容量:** `50`

### 数理モデル

上の問題の場合、数理モデルは以下のようになる。


**決定変数:**
決定変数はバイナリ変数である。
$$
x_0, x_1, x_2, x_3, x_4 \in \{0, 1\}
$$

以降のプログラムでは、決定変数の添え字は0から始まる点に注意せよ。

**目的関数:**
ナップザックに入れられたアイテムの総価値の最大化
$$
\text{Maximize} \quad Z = 60 x_0 + 100 x_1 + 120 x_2 + 80 x_3 + 90 x_4
$$

**制約条件:**
ナップザックに入れられたアイテムの総重量は、ナップザックの最大容量 $C$ を超えてはならない。
$$
10 x_0 + 20 x_1 + 30 x_2 + 15 x_3 + 25 x_4  \le C
$$


###問題のパラメータ設定

In [ ]:
import mip
import numpy as np

# 問題設定
# アイテムの価値のリスト
values = [60, 100, 120, 80, 90]
# アイテムの重さのリスト
weights = [10, 20, 30, 15, 25]
# ナップザックの容量
capacity = 50
# アイテム数
num_items = len(values)

print(f"アイテム数: {num_items}")
print(f"価値: {values}")
print(f"重さ: {weights}")
print(f"ナップザック容量: {capacity}")

次に、このデータを用いてナップザック問題を解く。

In [ ]:
# 1. モデルの作成
# モデルインスタンスを作成します。
# model = mip.Model(name="Knapsack_01")
model = mip.Model(name="Knapsack_01", sense=mip.MAXIMIZE) # 最大化問題として設定

# 2. 変数の定義
# 各アイテムを入れるかどうかを表すバイナリ変数をリストとして定義
# x[i] はアイテム i をナップザックに入れる場合に 1、入れない場合に 0 を取ります。
# 復習：連続変数の場合は、var_type=mip.CONTINUOUS であった
x = [model.add_var(var_type=mip.BINARY, name=f"x_{i}") for i in range(num_items)]
# x[0], x[1], x[2], x[3], x[4] と5個の変数リストとして定義された

###目的関数の設定

ナップザックに入れられたアイテムの総価値を最大化する。
$$
\text{Maximize} \quad Z = v_1 x_1 + v_2 x_2 + \cdots + v_N x_N = \sum_{i=1}^{N} v_i x_i
$$

In [ ]:
# 3. 目的関数の設定
# 目的は、選択されたアイテムの価値の合計を最大化すること
# 最大化には mip.maximize() を用いる
# mip.xsum() は、線形表現を効率的に生成するための関数です。
model.objective = mip.xsum(values[i] * x[i] for i in range(num_items))

In [ ]:
# print() を使い確認する
print( mip.xsum(values[i] * x[i] for i in range(num_items)) )

`mip.maximize()`を使い、最大化の目的関数を設定することも可能である。

In [ ]:
model.objective = mip.maximize(mip.xsum(values[i] * x[i] for i in range(num_items)))

###制約条件の設定

ナップザックに入れられたアイテムの総重量は、ナップザックの最大容量 $C$ を超えてはならないという制約条件を追加する。
$$
\sum_{i=1}^{N} w_i x_i \le C
$$
を追加する

In [ ]:
# 4. 制約条件の追加
# 選択されたアイテムの重さの合計がナップザックの容量を超えないようにします。
model.add_constr( mip.xsum(weights[i] * x[i] for i in range(num_items)) <= capacity, name="capacity_constraint")

# モデルの情報をファイルに書き出すことも可能です（デバッグなどに有用）。
# model.write("knapsack_model.lp") # LPファイル形式
# model.write("knapsack_model.mps") # MPSファイル形式

###最適化の実行

In [ ]:
# 5. 求解
# optimize() メソッドを呼び出して最適化を実行します。
# max_seconds パラメータで計算時間の上限（秒単位）を設定できます（任意）。
status = model.optimize(max_seconds=30.0)

###結果の表示

In [ ]:
# 6. 結果の表示と解釈
if status == mip.OptimizationStatus.OPTIMAL:
    print(f"最適化成功！")
    print(f"目的関数の最適値 (最大総価値): {model.objective_value:.2f}")

    selected_items_indices = []
    selected_items_values = []
    selected_items_weights = []
    total_weight = 0
    print("\n選択されたアイテム:")
    for i in range(num_items):
        # バイナリ変数の値は厳密に0または1にならない場合があるため、0.99以上などを選択基準とします。
        if x[i].x >= 0.99:
            selected_items_indices.append(i)
            selected_items_values.append(values[i])
            selected_items_weights.append(weights[i])
            total_weight += weights[i]
            print(f"  アイテム {i}: 価値={values[i]}, 重さ={weights[i]}")

    print(f"\n選択されたアイテムのインデックス: {selected_items_indices}")
    print(f"選択されたアイテムの総価値: {sum(selected_items_values):.2f}")
    print(f"選択されたアイテムの総重量: {total_weight:.2f} (容量: {capacity})")

elif status == mip.OptimizationStatus.FEASIBLE:
    print(f"実行可能解が見つかりましたが、最適解である保証はありません。")
    print(f"目的関数の値: {model.objective_value:.2f}")
elif status == mip.OptimizationStatus.INFEASIBLE:
    print("実行不可能: 制約を満たす解が存在しません。")
elif status == mip.OptimizationStatus.UNBOUNDED:
    print("非有界: 目的関数を無限に大きく（または小さく）できます。")
else:
    print(f"最適解は見つかりませんでした。ステータスコード: {status}")

**求解結果の解釈:**
上記のコードを実行すると、`python-mip` は最適解を探索します。
* `status` 変数には求解プロセスの結果（最適解が見つかったか、実行不可能かなど）が格納されます。
* `mip.OptimizationStatus.OPTIMAL` は最適解が見つかったことを意味します。
* `model.objective_value` には目的関数の最適値（この場合は最大化された総価値）が格納されます。
* 各決定変数 `x[i].x` には、その変数の最適解における値（この場合は0または1）が格納されます。

この例では、どのアイテムをナップザックに入れると総価値が最大になるか、そしてその時の総価値と総重量が表示されます。

## モデルの改善に関する視点

基本的な0-1ナップザック問題以外にも、さまざまなバリエーションや拡張が考えられる。

* **多次元ナップザック問題:** 重さだけでなく、体積など複数の制約がある場合。例えば、各アイテムに重さと体積があり、ナップザックには重さと体積それぞれの容量制限がある場合などである。この場合、制約条件が複数になる。
$$\sum_{i=1}^{N} w_{ij} x_i \le C_j \quad (\text{for each constraint } j)$$
ここで、$w_{ij}$ はアイテム $i$ の制約 $j$ に対する消費量（例：重さ、体積）、$C_j$ は制約 $j$ の容量である。

* **個数制限付きナップザック問題 (Bounded Knapsack Problem):** 各アイテムを複数個ナップザックに入れることができ、各アイテムの利用可能な個数に上限がある場合。この場合、決定変数は整数変数となり、各変数に上限が設定される。
$$x_i \in \{0, 1, \dots, u_i\}$$
ここで $u_i$ はアイテム $i$ の最大個数である。

* **複数ナップザック問題:** 複数のナップザックがあり、各アイテムをいずれかのナップザックに入れるか、どのナップザックにも入れないかを選択する問題。

これらのバリエーションも、決定変数や制約条件を適切に変更することで、`python-mip` を用いてモデル化し、解を求めることが可能である。モデルの複雑性が増すと計算時間も増加する傾向があるため、効率的な定式化や、場合によってはより強力な商用ソルバーの利用も検討される。

#演習問題


### 演習問題1：

上で扱った例題のアイテムデータとナップザック容量を変更して、
最適化せよ。最大総価値を解答する。


* **アイテム:**
    * 価値: `[45, 30, 50, 20, 65, 40, 25, 70]`
    * 重さ: `[5, 3, 6, 2, 7, 4, 3, 8]`
* **ナップザック容量:** `15`


### 演習問題2：

各アイテムは何個でも（ナップザックの容量が許す限り）選択できるナップザック問題を考える。これは、決定変数の型をバイナリ変数から整数変数に変更することで対応できる。なお、`python-mip` の変数の上限はデフォルトで無限大である。

* **アイテム:**
    * 価値: `[10, 40, 30, 50]`
    * 重さ: `[5, 4, 6, 3]`
* **ナップザック容量:** `10`


各アイテムを何個でも選択できるという条件の下で、総価値を最大化せよ。

この問題では、決定変数 $x_i$ はアイテム $i$ を選択する個数を表す非負整数となる。
$$x_i \ge 0, \text{integer}$$

`python-mip` では `var_type=mip.INTEGER` を指定する。
下限 `lb=0` を設定してもよいが、デフォルトで0なので明示しなくても良い。


### 演習問題3：

ある目標価値以上を達成するアイテムの組み合わせのうち、総重量が最小となるものを求める問題を考える。

* **アイテム:**
    * 価値: `[70, 80, 100, 60, 90]`
    * 重さ: `[10, 12, 18, 9, 14]`
* **目標総価値:** `200`

`python-mip` を用いてこの問題を解き、
総重量の最小値を求めよ。

**考え方:**
この問題では、目的関数が「総重量の最小化」に変わる。また、制約条件として「総価値が目標値以上である」という条件が加わる。


### 演習問題4：

特定のアイテムを選択する場合、別の特定のアイテムも必ず選択しなければならない、あるいは選択してはならない、といった依存関係があるナップザック問題を考える。
最大総価値を解答せよ。

* **アイテム:**
    * 価値: `[50, 80, 60, 70, 40]`
    * 重さ: `[5, 7, 4, 6, 3]`
* **ナップザック容量:** `15`
* **追加制約:**
    * アイテム0を選択する場合、アイテム2 も必ず選択しなければならない。
    * アイテム1とアイテム3は同時には選択できない（どちらか一方、またはどちらも選択しない）。
    
    ただし、アイテムは 0,1,2,3,4 と番号付けされている。


### 演習問題5：

アイテムには「重さ」と「体積」の2つの属性があり、ナップザックには「最大許容重量」と「最大許容体積」の2つの容量制限がある状況を考える。

* **アイテム:**
    * 価値:   `[100, 150, 80, 120, 90, 130]`
    * 重さ:   `[10,  15,  7,  12,  8,  14]`
    * 体積:   `[ 5,   8,  6,   7,  4,   9]`
* **ナップザック容量:**
    * 最大許容重量: `30`
    * 最大許容体積: `20`

この問題を解き、最大総価値を解答せよ。


### 応用問題6：

アイテムがいくつかのカテゴリに分類されており、各カテゴリから少なくとも1つのアイテムを選択しなければならない、という制約のあるナップザック問題を考える。

* **アイテム:**
    * 価値:      `[30, 20, 50, 70, 40, 60, 80]`
    * 重さ:      `[ 2,  3,  5,  6,  3,  4,  7]`
    * カテゴリID: `[ 0,  0,  1,  1,  1,  2,  2]`
* **ナップザック容量:** `15`
* **追加制約:**
    * カテゴリ0 (アイテム0, 1) から少なくとも1つ選択する。
    * カテゴリ1 (アイテム2, 3, 4) から少なくとも1つ選択する。
    * カテゴリ2 (アイテム5, 6) から少なくとも1つ選択する。

この問題を解き、最大総価値を解答せよ。
